# Flood Hotspot Mapping — Weighted FSI & Risk Map
**Study Area:** Nairobi Watershed, Kenya  
**Prerequisite:** Run `01_data_prep_and_modelling.ipynb` first to generate `model_state.pkl`

This notebook:
1. Loads normalised layers from notebook 1
2. Applies AHP weights to compute the Flood Susceptibility Index (FSI)
3. Classifies FSI into 5 zones (Very Low → Very High)
4. Generates the final flood risk map with hillshade overlay
5. Exports FSI rasters and per-class TIFs for QGIS overlay

In [ ]:
import os, pickle, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap, BoundaryNorm, LightSource
import rasterio
warnings.filterwarnings('ignore')

DATA_DIR = r'C:/Users/YourName/GEE_Exports'  # <-- CHANGE THIS

# Load state from notebook 1
with open(os.path.join(DATA_DIR, 'model_state.pkl'), 'rb') as f:
    state = pickle.load(f)

normalised  = state['normalised']
nodata_mask = state['nodata_mask']
ref_profile = state['ref_profile']
rasters     = state['rasters']
rows, cols  = state['rows'], state['cols']
print(f'Loaded. Grid: {rows} x {cols}')

In [ ]:
# ============================================================
# AHP WEIGHTS — based on flood literature for East Africa
# Weights sum to 1.0
# ============================================================
weights = {
    'rainfall':        0.2646,
    'elevation':       0.2255,
    'dist_river':      0.1598,
    'slope':           0.1211,
    'twi':             0.0904,
    'ndvi':            0.0591,
    'dist_impervious': 0.0467,
    'clay':            0.0328,
}
print(f'Weight sum: {sum(weights.values()):.4f}')

fsi = np.zeros((rows, cols), dtype='float32')
for layer, w in weights.items():
    fsi += normalised[layer] * w
fsi[nodata_mask] = np.nan

print(f'FSI range: {np.nanmin(fsi):.4f} to {np.nanmax(fsi):.4f}')
print(f'FSI mean : {np.nanmean(fsi):.4f}')

In [ ]:
# ============================================================
# CLASSIFY INTO 5 ZONES (percentile-based equal-frequency)
# ============================================================
breaks = np.nanpercentile(fsi, [0, 20, 40, 60, 80, 100])
labels = ['Very Low', 'Low', 'Moderate', 'High', 'Very High']

print('Class breaks:')
for i, label in enumerate(labels):
    print(f'  {label:<12}: {breaks[i]:.4f} - {breaks[i+1]:.4f}')

fsi_classes = np.full_like(fsi, np.nan)
for i in range(5):
    mask = (fsi >= breaks[i]) & (fsi < breaks[i+1]) if i < 4 else (fsi >= breaks[i]) & (fsi <= breaks[i+1])
    fsi_classes[mask] = i + 1
fsi_classes[nodata_mask] = np.nan

# Area statistics
total_valid = np.sum(~nodata_mask)
print('\nFlood Susceptibility Zone Statistics:')
print(f'{"Zone":<14} {"Area (km2)":>12} {"Coverage":>10}')
print('-' * 40)
for i, label in enumerate(labels, 1):
    count    = np.sum(fsi_classes == i)
    area_km2 = count * 30 * 30 / 1e6
    pct      = 100 * count / total_valid
    print(f'{label:<14} {area_km2:>12.2f} {pct:>9.1f}%')

In [ ]:
# ============================================================
# FLOOD RISK MAP
# ============================================================
class_colors = ['#1a9850', '#91cf60', '#ffffbf', '#fc8d59', '#d73027']
cmap5  = ListedColormap(class_colors)
norm5  = BoundaryNorm([0.5,1.5,2.5,3.5,4.5,5.5], cmap5.N)

elev_hs = rasters['elevation'].copy()
elev_hs[np.isnan(elev_hs)] = np.nanmean(elev_hs)
shade = LightSource(azdeg=315, altdeg=45).hillshade(elev_hs, vert_exag=2, dx=30, dy=30)

fig, axes = plt.subplots(1, 2, figsize=(18, 10), facecolor='white')

# Left: continuous FSI
axes[0].set_facecolor('#d6eaf8')
axes[0].imshow(shade, cmap='gray', alpha=0.4)
im1 = axes[0].imshow(fsi, cmap='RdYlGn_r',
    vmin=np.nanpercentile(fsi, 2), vmax=np.nanpercentile(fsi, 98), alpha=0.85)
plt.colorbar(im1, ax=axes[0], fraction=0.035, pad=0.02, shrink=0.80, label='Flood Susceptibility Index')
axes[0].contour(nodata_mask.astype(float), levels=[0.5], colors=['#222222'], linewidths=[1.0])
axes[0].set_title('Flood Susceptibility Index\n(Continuous Weighted Overlay)', fontsize=12, fontweight='bold')
axes[0].axis('off')

# Right: classified zones
axes[1].set_facecolor('#d6eaf8')
axes[1].imshow(shade, cmap='gray', alpha=0.35)
axes[1].imshow(fsi_classes, cmap=cmap5, norm=norm5, alpha=0.88)
axes[1].contour(nodata_mask.astype(float), levels=[0.5], colors=['#222222'], linewidths=[1.0])
axes[1].set_title('Flood Hotspot Zones — Nairobi Watershed\n(5 Susceptibility Classes)', fontsize=12, fontweight='bold')
axes[1].axis('off')

# Legend with area stats
patches = []
for i, (color, label) in enumerate(zip(class_colors, labels), 1):
    area_km2 = np.sum(fsi_classes == i) * 30 * 30 / 1e6
    pct = 100 * np.sum(fsi_classes == i) / total_valid
    patches.append(mpatches.Patch(facecolor=color, edgecolor='#555', linewidth=0.5,
        label=f'{label}  —  {area_km2:.1f} km2  ({pct:.1f}%)'))
axes[1].legend(handles=patches, loc='lower left', fontsize=9, framealpha=0.93,
    title='Class  |  Area  |  Coverage', title_fontsize=8.5)

# Weight annotation
wt = 'AHP Weights\n' + chr(8212)*18 + '\n' + '\n'.join([f'{k:<18} {v:.4f}' for k, v in weights.items()])
axes[1].text(0.99, 0.99, wt, transform=axes[1].transAxes, fontsize=7.5, va='top', ha='right',
    fontfamily='monospace', bbox=dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='#aaa', alpha=0.93))

fig.suptitle('Flood Susceptibility Model — Nairobi Watershed, Kenya  (2022-2024)',
    fontsize=14, fontweight='bold', y=1.005)
plt.tight_layout(pad=2.5)
plt.savefig(os.path.join(DATA_DIR, 'flood_hotspot_map_final.png'), dpi=200, bbox_inches='tight')
plt.show()
print('Saved: flood_hotspot_map_final.png')

In [ ]:
# ============================================================
# EXPORT ALL GEOTIFF OUTPUTS
# ============================================================
def save_tif(array, path, dtype='float32', nodata=-9999):
    meta = ref_profile.copy()
    meta.update({'count':1,'dtype':dtype,'nodata':nodata})
    out = array.copy(); out[np.isnan(out)] = nodata
    with rasterio.open(path, 'w', **meta) as dst:
        dst.write(out.astype(dtype), 1)
    print(f'  Saved: {os.path.basename(path)}')

print('Exporting rasters...')
save_tif(fsi,        os.path.join(DATA_DIR, 'FSI_continuous_nairobi.tif'))
save_tif(fsi_classes,os.path.join(DATA_DIR, 'FSI_classes_final_nairobi.tif'), dtype='uint8', nodata=0)

# High risk only
save_tif(np.where((fsi_classes >= 4) & (~nodata_mask), 1.0, np.nan),
    os.path.join(DATA_DIR, 'FSI_high_risk_only_nairobi.tif'))

# Individual class TIFs
for cls, name in enumerate(['very_low','low','moderate','high','very_high'], 1):
    arr = np.where((fsi_classes == cls) & (~nodata_mask), float(cls), np.nan).astype('float32')
    save_tif(arr, os.path.join(DATA_DIR, f'FSI_class_{name}_nairobi.tif'))

print('\nAll outputs ready for QGIS overlay.')
print('Next step: load FSI_classes_final_nairobi.tif in QGIS and run Polygonize.')